In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional


In [2]:
# Load dataset
df = pd.read_csv("NIFTY50.csv")

# Ensure correct column usage
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

data = df[['Close']]
data.dropna(inplace=True)

print(data.head())


            Close
Date             
2009-03-02  43.17
2009-03-03  43.89
2009-03-04  42.52
2009-03-05  41.49
2009-03-06  38.16


/var/folders/z4/58kf2rh54s395f0pkgk591q00000gn/T/ipykernel_979/547325863.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.dropna(inplace=True)


In [3]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)


In [4]:
def create_sequences(data, window_size=60):
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i - window_size:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

WINDOW_SIZE = 60

X, y = create_sequences(scaled_data, WINDOW_SIZE)

# Reshape for Bi-LSTM input
X = X.reshape(X.shape[0], X.shape[1], 1)


In [5]:
train_size = int(0.8 * len(X))

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]


In [6]:
model = Sequential()

model.add(
    Bidirectional(
        LSTM(50, return_sequences=True),
        input_shape=(X_train.shape[1], 1)
    )
)
model.add(Dropout(0.2))

model.add(
    Bidirectional(
        LSTM(50)
    )
)
model.add(Dropout(0.2))

model.add(Dense(1))


/Users/krish/stock_ml/venv/lib/python3.11/site-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [7]:
model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 60, 100)        │        20,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 100)            │        60,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,301 (317.58 KB)

 Trainable params: 81,301 (317.58 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test)
)


Epoch 1/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.0015 - val_loss: 0.0032
Epoch 2/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 7.0972e-04 - val_loss: 0.0026
Epoch 3/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 6.4299e-04 - val_loss: 0.0017
Epoch 4/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 5.9937e-04 - val_loss: 0.0017
Epoch 5/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 5.1472e-04 - val_loss: 0.0012
Epoch 6/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 4.8461e-04 - val_loss: 0.0013
Epoch 7/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 5.1299e-04 - val_loss: 0.0012
Epoch 8/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 5.2483e-04 - val_loss: 9.5209e-04
Epoch 9/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 4.6828e-04 - val_loss: 9.3646e-04
Epoch 10/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 4.6837e-04 - val_loss: 0.0012
Epoch 11/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 4.2662e-04 - val_loss: 9.3767e-04
Epoch 12/20

In [9]:
predicted_prices = model.predict(X_test)

# Inverse scale
predicted_prices = scaler.inverse_transform(predicted_prices.reshape(-1, 1))
actual_prices = scaler.inverse_transform(y_test.reshape(-1, 1))


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


In [11]:
rmse = np.sqrt(mean_squared_error(actual_prices, predicted_prices))
mae  = mean_absolute_error(actual_prices, predicted_prices)

print("RMSE:", rmse)
print("MAE :", mae)


RMSE: 2.1976732524297997
MAE : 1.128886037309237


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(actual_prices, label="Actual Price")
plt.plot(predicted_prices, label="Predicted Price")
plt.title("Bi-LSTM Prediction on NIFTY 50")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.show()


In [ ]:
last_60_days = scaled_data[-60:]
last_60_days = last_60_days.reshape(1, 60, 1)


In [ ]:
next_day_scaled = model.predict(last_60_days)


In [ ]:
next_day_price = scaler.inverse_transform(next_day_scaled)

print("Predicted NIFTY 50 closing price for next day:")
print(next_day_price[0][0])
